### Service code 를 주피터에서 실행될수 있도록 수정한 코드
실행방법 : 코드를 실행하고 다음 셀에서 정수를 입력하고 실행한다.

In [ ]:
# import modules
import rclpy
from rclpy.node import Node
from example_interfaces.srv import AddTwoInts

from rclpy.executors import SingleThreadedExecutor

import threading
import time

In [ ]:
# Server Node
class MinimalService(Node):

    def __init__(self):
        super().__init__("minimal_service")
        self.srv = self.create_service(AddTwoInts, "add_two_ints", self.add_two_ints_callback)

    def add_two_ints_callback(self, request, response):
        self.get_logger().info(f"Incoming request: {request.a} + {request.b}")
        response.sum = request.a + request.b
        return response

In [ ]:
# Client Node
class MinimalClientAsync(Node):

    def __init__(self):
        super().__init__("minimal_client_async")
        self.cli = self.create_client(AddTwoInts, "add_two_ints")
        while not self.cli.wait_for_service(timeout_sec=1.0):
            self.get_logger().info("Service not available, waiting again...")
        self.req = AddTwoInts.Request()

    def send_request(self, a, b):
        self.req.a = a
        self.req.b = b
        return self.cli.call_async(self.req)

In [ ]:
# 전역에서 init()은 한 번만 호출
print(rclpy.ok())
if not rclpy.ok():
    rclpy.init()

In [ ]:
# executor 실행
executor = SingleThreadedExecutor()

In [ ]:
# Start the server node
server_node = MinimalService()
executor.add_node(server_node)

In [ ]:

def run_client(a, b):
    client_node = MinimalClientAsync()
    future = client_node.send_request(a, b)
    executor.add_node(client_node)

    def spin_until_future_complete():
        while rclpy.ok() and not future.done():
            executor.spin_once(timeout_sec=0.1)
            time.sleep(0.1)

    # spin을 백그라운드에서 실행
    spin_thread = threading.Thread(target=spin_until_future_complete)
    spin_thread.start()
    spin_thread.join()

    if future.done():
        response = future.result()
        client_node.get_logger().info(
            f"Result of add_two_ints: for {a} + {b} = {response.sum}"
        )
        print(f"{a} + {b} = {response.sum}")

    # 종료 처리(client)
    executor.remove_node(client_node)
    client_node.destroy_node()

In [ ]:
# 실행 예시
run_client(3, 7)

In [ ]:
# 실행 예시
run_client(10, 15)

In [ ]:
# 종료 처리(server)
executor.remove_node(server_node)
server_node.destroy_node()
rclpy.shutdown()